In [ ]:
import os
import torch
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm

# CONFIG
frame_root = "./data/video/"         # folder with subfolders: video1/, video2/, ...
output_dir = "./data/video_tensors/" # where to save .pt files
target_size = (224, 224)

# TRANSFORM
transform = T.Compose([
    T.Resize(target_size),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet-style
])

# CREATE OUTPUT DIR
os.makedirs(output_dir, exist_ok=True)

# PROCESS EACH VIDEO FOLDER
video_folders = sorted([f for f in os.listdir(frame_root) if os.path.isdir(os.path.join(frame_root, f))])

for folder in tqdm(video_folders, desc="Preprocessing videos"):
    fpath = os.path.join(frame_root, folder)
    frames = sorted([f for f in os.listdir(fpath) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
    tensor_list = []

    for fname in frames:
        try:
            img = Image.open(os.path.join(fpath, fname)).convert("RGB")
            tensor = transform(img)
            tensor_list.append(tensor)
        except Exception as e:
            print(f"Error in {fname}: {e}")

    if tensor_list:
        vid_tensor = torch.stack(tensor_list)  # (T, C, H, W)
        torch.save(vid_tensor, os.path.join(output_dir, f"{folder}.pt"))
